# Gaussian Mixture Models

**Companion lesson:** https://ml-viz.vercel.app/courses/probabilistic-models/01-gaussian-mixture-models

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Soft vs Hard Clustering

GMMs assign **probabilities** (responsibilities) to each cluster, unlike K-Means which makes hard assignments.

In [ ]:
np.random.seed(42)
n = 150
X = np.concatenate([np.random.randn(n) * 0.8 - 2, np.random.randn(n) * 1.2 + 3])

# Simple EM for 1D GMM
K = 2
mu = np.array([-3.0, 2.0])
sigma = np.array([1.0, 1.0])
pi = np.array([0.5, 0.5])

for _ in range(50):
    # E-step
    gamma = np.zeros((len(X), K))
    for k in range(K):
        gamma[:, k] = pi[k] * norm.pdf(X, mu[k], sigma[k])
    gamma /= gamma.sum(axis=1, keepdims=True)
    # M-step
    Nk = gamma.sum(axis=0)
    for k in range(K):
        mu[k] = np.sum(gamma[:, k] * X) / Nk[k]
        sigma[k] = np.sqrt(np.sum(gamma[:, k] * (X - mu[k])**2) / Nk[k])
        pi[k] = Nk[k] / len(X)

# Visualize responsibilities
fig, axes = plt.subplots(2, 1, figsize=(10, 7), gridspec_kw={'height_ratios': [1, 2]})

axes[0].scatter(X, np.zeros_like(X), c=gamma[:, 0], cmap='coolwarm', s=20, alpha=0.8)
axes[0].set_title('Responsibility P(cluster=0) per point', color='white', fontsize=11)
axes[0].set_yticks([])

x_grid = np.linspace(-6, 8, 300)
mixture = sum(pi[k] * norm.pdf(x_grid, mu[k], sigma[k]) for k in range(K))
axes[1].hist(X, bins=50, density=True, color='#818cf8', alpha=0.4, edgecolor='#1a1d27')
axes[1].plot(x_grid, mixture, color='#14b8a6', linewidth=2, label='Mixture')
for k in range(K):
    axes[1].plot(x_grid, pi[k] * norm.pdf(x_grid, mu[k], sigma[k]), '--',
                 color=['#f43f5e', '#eab308'][k], linewidth=1.5,
                 label=f'Component {k+1}: μ={mu[k]:.2f}, σ={sigma[k]:.2f}, π={pi[k]:.2f}')
axes[1].set_title('Gaussian Mixture Fit', color='white', fontsize=11)
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

## Responsibility by hand

The responsibility $\gamma_{ik}=P(z_i{=}k\mid x_i)$ is just Bayes' theorem: prior $\pi_k$ times likelihood $\mathcal N(x_i\mid\mu_k,\sigma_k)$, normalized over components. Below we reproduce the lesson's worked example ($x=2$, two unit Gaussians at $\mu=0,5$) and confirm $\gamma=(0.924, 0.076)$.

In [ ]:
import numpy as np
from scipy.stats import norm

# Worked example: x=2, two components N(0,1) and N(5,1), equal weights
x = 2.0
pi = np.array([0.5, 0.5])
mu = np.array([0.0, 5.0])
sig = np.array([1.0, 1.0])

# Gaussian densities (the 1/sqrt(2pi) cancels in the ratio, kept for clarity)
dens = norm.pdf(x, mu, sig)
print('N(2|0,1) =', round(dens[0], 4), '  N(2|5,1) =', round(dens[1], 5))

# Bayes' theorem -> responsibilities
unnorm = pi * dens
gamma = unnorm / unnorm.sum()
print('responsibilities gamma =', np.round(gamma, 3), ' (lesson: [0.924, 0.076])')

# A point exactly between the means splits 50/50
g_mid = (pi * norm.pdf(2.5, mu, sig)); g_mid /= g_mid.sum()
print('at x=2.5 (midpoint):', np.round(g_mid, 3))

# Sanity: M-step mu is the responsibility-weighted mean (soft K-Means).
# Two points, full E then M on mu, to show the weighting in action.
Xp = np.array([2.0, 2.5])
G = np.array([(pi * norm.pdf(xi, mu, sig)) for xi in Xp])
G /= G.sum(axis=1, keepdims=True)
Nk = G.sum(axis=0)
mu_new = (G * Xp[:, None]).sum(axis=0) / Nk
print('M-step mu (responsibility-weighted mean) =', np.round(mu_new, 3))


## 2D GMM with Elliptical Clusters

Unlike K-Means, GMMs can model **elliptical** clusters with different orientations.

In [ ]:
np.random.seed(42)
n = 200

# Create clusters with different shapes
angle = np.pi / 4
R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
X1 = R @ np.diag([3, 0.5]) @ np.random.randn(2, n // 2) + np.array([0, 0])
X2 = np.random.randn(2, n // 2) * 1.5 + np.array([8, 2])
X_2d = np.vstack([X1.T, X2])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# K-Means would force circular clusters
from sklearn.cluster import KMeans
km = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_2d)
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=km.labels_, cmap='viridis', s=10, alpha=0.6)
axes[0].set_title('K-Means (spherical clusters)', color='white', fontsize=11)

# GMM can model ellipses
from sklearn.mixture import GaussianMixture
gmm = GaussianMixture(n_components=2, covariance_type='full', random_state=42).fit(X_2d)
labels = gmm.predict(X_2d)
probs = gmm.predict_proba(X_2d)
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=probs[:, 0], cmap='coolwarm', s=10, alpha=0.7)
axes[1].set_title('GMM (elliptical clusters, soft)', color='white', fontsize=11)

for ax in axes:
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Covariance type and model selection

GMM covariance can be `full`, `tied`, `diag`, or `spherical` — more flexibility means more parameters. **BIC** picks the number of components by penalizing complexity.

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs

X, _ = make_blobs(n_samples=500, centers=3, cluster_std=1.0, random_state=0)
bics = []
for k in range(1, 7):
    gm = GaussianMixture(n_components=k, covariance_type='full', random_state=0).fit(X)
    bics.append(gm.bic(X))
    print(f'K={k}: BIC = {gm.bic(X):.0f}')
print('best K by BIC:', int(np.argmin(bics)) + 1)

## Key takeaways

- A GMM models data as a weighted mix of Gaussians — **soft** assignments (responsibilities).
- Unlike K-Means it captures **elliptical** clusters and gives probabilistic membership.
- **Covariance type** trades flexibility for parameters; `full` is most expressive.
- Choose the number of components with **BIC/AIC**, not the likelihood alone.